## Lab 04

## Zadanie 1 (Igor Sieradzki)

Funkcja <t>flow_rate</i> do modyfikacji

In [1]:
import epsilon


def flow_rate(weight, time, *, period = 60 , units_per_kg = 1):
    """ Funkcja wylicza ile wagi produktu przybyło/ubyło w jednostce czasu """
    return weight / time * period * units_per_kg

weight = 0.5
time = 3

In [2]:
flow = flow_rate(weight, time)
print("{0:.3} kg per second".format(flow))

10.0 kg per second


In [ ]:
Zmodyfikuj funckje <i>flow_rate</i> tak, aby poniższe wywołania działały poprawnie.

In [3]:
flow = flow_rate(weight, time, period=60, units_per_kg=1000)
print("{} grams per minute".format(flow))

10000.0 grams per minute


In [4]:
flow = flow_rate(weight, time, period=1, units_per_kg=1)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [5]:
flow = flow_rate(weight, time)
print("{0:.3} grams per minute".format(flow))

10.0 grams per minute


Wyjaśnić czemu poniższe rzucanie błędu jest porządane

In [6]:
try:
    flow = flow_rate(weight, time, 3600, 2.2)
except TypeError:
    print(True)

True


## Zadanie 2 (Igor Sieradzki)

Dopisz definicje dekoratora _timeit_, który wypisze na wyjscie standardowe czas wywołania udekorowanej funckji np.

```
@timeit
def foo(x):
    return x**2
    
r = foo(2)
```
wypize: <br>
    `Function foo took: 0.00001 seconds` <br>
oraz zwróci 4

In [9]:
from time import time

def timeit(func):
    """ Wypisuje czas wywołania udekorowanej funckji """
    def wrapper(*args, **kwargs):
        start = time()
        result = func(*args, **kwargs)
        print("Time:", time() - start, "seconds")
        return result

    return wrapper

@timeit
def squares_list(n):
    squares = []
    for i in range(n):
        squares.append(i ** 2)
    return squares

@timeit
def squares_comprehension(n):
    return [i ** 2 for i in range(n)]

@timeit
def squares_map(n):
    return map(lambda x: x**2, range(n))

n = 1000000
l = squares_list(n)
c = squares_comprehension(n)
m = squares_map(n)

Time: 0.09347009658813477 seconds
Time: 0.06722497940063477 seconds
Time: 5.9604644775390625e-06 seconds


## Zadanie 3 (Igor Sieradzki)

Dopisz definicje dekoratora _derivate_ wg. instrukcji w _docstringu_

In [13]:
import sys
sys.float_info.epsilon# epsilon maszynowy


def derivate(epsilon=None):
    """
    Zwraca pochodną funkcji w punkcie, wg. wzoru f'(x) = [f(x+h) - f(x)]/h,
    gdzie h jest parametrem dekoratora, jeśli nie zostanie podany, należy przyjąć 1000 * epsilon maszynowy
    """
    if epsilon is None:
        epsilon = 1000 * sys.float_info.epsilon

    def decorator(func):
        def wrapper(x):
            return (func(x + epsilon) - func(x)) / epsilon

        return wrapper

    return decorator



@derivate(0.01)
def f(x):
    return x*x

@derivate(0.00001)
def g(x):
    return x*x*x+3

def test(a, b, eps=1):
    return abs(round(a)-round(b)) < eps

print(test(f(100), 200.0))
print(round(f(0)) == 0.0)

print(test(g(100), 30000.0))
print(round(g(0)) == 0.0)

from random import random
for x in [random()*1000. for _ in range(20)]:
    print(f(x), 2*x, '\t', test(f(x), 2*x))
    print(g(x), 3*x**2, '\t', test(g(x), 3*x**2))

True
True
True
True
242.23879200580996 242.22879200566206 	 True
44006.0944063589 44006.09075739172 	 True
1228.8435610651504 1228.833561068823 	 True
1132523.9539146423 1132523.9406068136 	 True
1733.77051576972 1733.7605157652702 	 True
2254444.15807724 2254444.1445199917 	 True
1038.383899780456 1038.3738997820797 	 True
808665.2815341948 808665.2668114833 	 True
464.9993707345857 464.9893707354411 	 True
162161.34317219257 162161.33617270616 	 True
1235.2489245473407 1235.2389245506993 	 True
1144361.4155054092 1144361.4005438762 	 True
737.8614382178057 737.8514382175645 	 True
408318.5695111751 408318.55865979625 	 True
1883.154639939312 1883.1446399433507 	 True
2659675.3120422363 2659675.301210529 	 True
1885.330646915827 1885.3206469115019 	 True
2665825.4742622375 2665825.456253103 	 True
338.1732711321092 338.1632711326883 	 True
85765.80360531807 85765.79845737005 	 True
391.8763101908553 391.86631019101515 	 True
115169.40975561737 115169.40379704066 	 True
314.98783766110

## Zadanie 4 (Igor Sieradzki)

Dopisz definicje dekoratora _accepts_ wg. instrukcji w _docstringu_

In [15]:
from inspect import signature

def accepts(*types):
    """Sprawdza czy udekorowanej funckji zostały podane odpowiednie parametry zdefiniowane
       w argumentach dekoratora"""
    def decorator(func):
        def wrapper(*args, **kwargs):
            bound = signature(func).bind(*args, **kwargs)

            for value, expected in zip(bound.arguments.values(), types):
                if not isinstance(value, expected):
                    raise TypeError("Wrong argument type")

            return func(*args, **kwargs)

        return wrapper

    return decorator

@accepts(str)
def capitalize(word):
    return word[0].upper() + word[1:]

print(capitalize('ola') == 'Ola')

try:
    capitalize(2)
except TypeError:
    print(True)

@accepts(float, int)
def static_pow(base, exp):
    return base ** exp 

print(static_pow(2., 2) == 4.)
print(static_pow(2., exp=2) == 4.)
print(static_pow(base=2., exp=2) == 4.)

try:
    static_pow('x', 10)
except TypeError:
    print(True)
    
try:
    static_pow(2, 2.2)
except TypeError:
    print(True)

True
True
True
True
True
True
True


## Zadanie 5 (Igor Sieradzki)

Dopisz definicje dekoratora _returns_ wg. instrukcji w _docstringu_

In [17]:
def returns(*types):
    """Sprawdza czy udekorowana funkcja zwraca poprawne argumenty, zdefiniowane w parametrach dekoratora"""
    def decorator(func):
        def wrapper(*args, **kwargs):
            result = func(*args, **kwargs)

            if len(types) == 1:
                values = (result,)
            else:
                if not isinstance(result, tuple):
                    raise TypeError("Expected a tuple")
                values = result

            if len(values) != len(types):
                raise TypeError("Incorrect number of returned values")

            for value, expected in zip(values, types):
                if not isinstance(value, expected):
                    raise TypeError("Incorrect return type")

            return result

        return wrapper

    return decorator

@returns(str)
def str_only_identity(word):
    return word

print(str_only_identity('hello') == 'hello')

try:
    str_only_identity(10)
except TypeError:
    print(True)
    
@returns(int, int)
def split_indices(x):
    return x[0], x[1]

print(split_indices(x=[6,9]) == (6,9))

try:
    split_indices('AB')
except TypeError:
    print(True)

True
True
True
True


## Zadanie 6 (Igor Sieradzki)
Stwórz dekorator cached służący do cachowania wywołań dowolnej funkcji, tzn. chcemy by:
* wywołanie funkcji z określonymi argumentami miało miejsce tylko raz
* funkcja mogła przyjmować dowolną liczbę nazwanych i nienazwanych argumentów
* nie musi reagować poprawnie na domyślne argumenty, tzn. wywołanie funkcji z domyślnymi argumentami a podanie dokładnie takich samych może być traktowane jako dwa różne wywołania
* na opakowanej funkcji można wywołać `.cache_reset()`, który usunie cache z pamięci
* wywołanie `.cache_status()` zwraca string z opisem w postaci: <br>
    `Function FUNCTION_NAME called X times, evaluated Y times`

In [20]:
from random import random

def cached(func):
    cache = {}
    called = 0
    evaluated = 0

    def wrapper(*args, **kwargs):
        nonlocal called, evaluated
        called += 1

        key = (args, tuple(sorted(kwargs.items())))

        if key not in cache:
            cache[key] = func(*args, **kwargs)
            evaluated += 1

        return cache[key]

    def cache_status():
        return (
            f"Function {func.__name__} called {called} times, "
            f"evaluated {evaluated} times"
        )

    wrapper.cache_reset = cache.clear
    wrapper.cache_status = cache_status

    return wrapper


@cached
def foo(x, y=1, z=4):
    return random()
        
print(foo(3) == foo(3))
print(foo(4) == foo(4))
print(foo(3, z=-1, y=3) == foo(3, y=3, z=-1))
print(foo(3) != foo(x=3))
a = foo(3)
foo.cache_reset()
print(a != foo(3))
print(foo.cache_status() == 'Function foo called 10 times, evaluated 5 times')

True
True
True
True
True
True


## Zadanie 7 (Krzysztof Hajto)

Napisz dekorator który będzie robić n-krotne złożenie funkcji, gdzie n jest parametrem dekoratora

In [ ]:
def zlozenie(n):
    def decorator(func):
        def wrapper(x):
            for _ in range(n):
                x = func(x)
            return x

        return wrapper

    return decorator

@zlozenie(3)
def f1(x):
    return x+1

@zlozenie(2)
def f2(x):
    return x*x

@zlozenie(5)
def f3(word):
    return "".join(chr(ord(l)+1) for l in word)

print(f1(2)==5)
print(f2(3)==81)
print(f3("alamakota")=="fqfrfptyf")

## Zadanie 8 (Krzysztof Hajto)

Python nie ma wbudowanej instrukcji switch. Ale posiada anonimowe funkcje oraz słowniki. Zaimplementuj poniższy switch w postaci słownika funkcji.
`
int my_function(x, y) {


    switch(x) {
        case 1: return y*y;
        case 2: return x+y;
        case 3: return x*y;
        case 4: return 0;
    }
}
`

PS. Nigdy nie róbcie tego w faktycznym kodzie :)

In [21]:
def my_function(x, y):
    operations = {
        1: lambda: y * y,
        2: lambda: x + y,
        3: lambda: x * y,
        4: lambda: 0
    }

    return operations[x]()

print(my_function(1,3)==9)
print(my_function(2,4)==6)
print(my_function(3,1)==3)
print(my_function(4,9)==0)

True
True
True
True
